# TensorFlow track — exercise solutions

Solutions are kept separate from lessons. Run the matching lesson first and use its dataset path and class mapping.

## TF00 — inspect and balance Homer/Bart dataset

In [ ]:
from pathlib import Path
from collections import Counter
root=Path('data/homer_bart_2/training_set')
counts={p.name:sum(1 for f in p.rglob('*') if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}) for p in root.iterdir() if p.is_dir()}
print(counts)
# Imbalance ratio = max(class_count)/min(class_count); inspect before deciding on class weights.
ratio=max(counts.values())/min(counts.values())
print('Imbalance ratio:',ratio)

## TF00 — compute class weights from training labels

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
# Use labels from the training split only.
y_train=np.concatenate([y.numpy().ravel() for _,y in train_ds]).astype(int)
classes=np.unique(y_train)
weights=compute_class_weight(class_weight='balanced',classes=classes,y=y_train)
class_weight={int(c):float(w) for c,w in zip(classes,weights)}
print(class_weight)
# Pass class_weight=class_weight to model.fit if imbalance justifies it.

## TF01 — gradual fine-tuning checklist

In [ ]:
# After frozen-backbone training, unfreeze the final residual stage only.
for layer in base.layers:
    layer.trainable = layer.name.startswith('conv5_') and not isinstance(layer,tf.keras.layers.BatchNormalization)
# Recompile with a low learning rate after changing trainable flags.
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='binary_crossentropy',metrics=['accuracy'])
print('Trainable layers:',sum(int(layer.trainable) for layer in base.layers))

## TF02 — PSNR and SSIM

In [ ]:
import tensorflow as tf
# Inputs are float images in [0,1], same shape.
psnr=tf.image.psnr(xf_test[:16],xf_recon,max_val=1.0)
ssim=tf.image.ssim(xf_test[:16],xf_recon,max_val=1.0)
print('Mean PSNR:',float(tf.reduce_mean(psnr)),'Mean SSIM:',float(tf.reduce_mean(ssim)))

## TF03 — fixed noise and model saving

In [ ]:
fixed_noise=tf.random.normal((16,100))
# Generate from the same latent batch after each epoch for meaningful visual comparison.
# Save checkpoints using generator.save('generator.keras') and discriminator.save('discriminator.keras').

## Answer-writing checklist

For each experiment state: hypothesis, one controlled variable, dataset split, seed, metrics, visual inspection, observed failure modes and conclusion. Do not write expected scores as measured results.